## Kinesis Data Streams

When data must feed **multiple downstream consumers**, each at its own pace, with the ability to **rewind and replay** — that's a **stream, not a queue**. **Kinesis Data Streams** is the AWS-native streaming service: producers push records; multiple consumers read them independently; records are retained **up to a year**. The difference from SQS is the point — SQS deletes a message after one consumer processes it; Kinesis **keeps records after consumption**, so many consumers read the same data and can reprocess. SQS suits async jobs; Kinesis suits **telemetry, clickstreams, IoT, change-capture** — anything with multiple consumers or reprocessing.

A stream is divided into **shards**, the unit of capacity: each shard does **1 MB/s or 1,000 records/s write** and **2 MB/s read** (shared across standard consumers). Scale by **splitting** (add shards) or **merging** (remove them). Every record carries a **partition key** that Kinesis hashes to pick a shard — same key → same shard → **ordered within that shard** (so choose a high-cardinality key to spread load, like the partitions lesson in DynamoDB).

Two **consumer models** with very different throughput. **Standard** consumers poll `GetRecords` and **share** the shard's 2 MB/s (five consumers → 400 KB/s each), ~200 ms latency. **Enhanced Fan-Out (EFO)** gives each consumer its **own dedicated 2 MB/s** via HTTP/2 push, ~70 ms latency — costs more but right when several consumers each need full throughput. Rule: **1–2 consumers, cost matters → standard; many consumers or low-latency → EFO.**